In [ ]:
require(data.table)
require(tidyverse)
require(phyloseq)
require(genefilter)
require(ggplot2)
require(RColorBrewer)
require(metacoder)
require(vegan)
require(DESeq2)
options(repr.plot.width=20, repr.plot.height=15)

## phyloseq cleanup

In [ ]:
#removing any taxa that don't show up in any samples to speed up the process
ps <- prune_taxa(taxa_sums(ps) > 0, ps)

#normalizing ps by converting rawcounts into relative abundances
#so samples with more reads wont be over represented
#using ps bc only to the count data (OTU table), while preserving the rest of the object
ps_norm = transform_sample_counts(ps, function(x) 1E6 * x / sum(x))

In [ ]:
#isolate just bacteria
ps_norm_bac=subset_taxa(ps_norm, Kingdom=="Bacteria")
#remove chloroplast order
ps_norm_nochlo=subset_taxa(ps_norm_bac, Order!="Chloroplast")
#remove mitochondria family
ps_norm_nomit=subset_taxa(ps_norm_nochlo, Family!="Mitochondria")
ps_norm_nomit

## new repeated colonies only dataframe

In [ ]:
# Return names which have more than one row of data
# Now filter
colonies <- sample_nomit %>%
  group_by(colony) %>%
  filter(n() != 1) %>%
  ungroup()

#how many colonies are represented?
nrow(sample_nomit)
nrow(colonies)

In [ ]:
# Restore rownames
rownames(colonies) <- colonies$SampleID
#check
head(rownames(colonies))
NROW(sample_names(ps_norm_nomit))

In [ ]:
colonies <- as.data.frame(colonies)
class(colonies)

In [ ]:
#create a new phyloseq object with colonies dataframe
col_clean <- phyloseq::sample_data(colonies)
sample_data(ps_norm_nomit) <- col_clean
head(sample_data(ps_norm_nomit))

# How are the microbiomes of individual coral colonies changing overtime?
- do they recruit fewer bacteria?
- do they shift to recruiting new bacteria?

In [ ]:
## make sure dates are in chronological order

# 2. reorder MonthYear as a factor in chronological order
sample_nomit$Month_year <- factor(sample_nomit$Month_year,levels = unique(sample_nomit$Month_year))

# How do the microbiomes of individual colonies change with exposure to stressors like heat and disease?